# Regression by Minimizing Absolute Deviation — Complete Experimental Workflow

This notebook runs the project from data preparation through OLS/LAD fitting, controlled experiments, validation, focused visualizations, and generated outputs.

The four retained visualizations are:
1. OLS vs LAD under contaminated responses.
2. SSE vs contamination.
3. SAE vs contamination.
4. HBK multivariate residual comparison.


## Regression objectives

For residuals

$$
r_i = y_i - x_i^T\beta,
$$

Ordinary Least Squares minimizes

$$
\min_{\beta}\sum_{i=1}^{n} r_i^2,
$$

while Least Absolute Deviations minimizes

$$
\min_{\beta}\sum_{i=1}^{n}|r_i|.
$$


In [ ]:
import pandas as pd
from IPython.display import Image, display

from absolute_deviation.data import DATASETS, load_dataset
from absolute_deviation.experiments import (
    FIGURE_DIR,
    RESULT_DIR,
    run_contamination_experiment,
    run_error_distribution_experiment,
    run_original_data,
    run_runtime_benchmark,
    validate_lad_solver,
)
from absolute_deviation.plotting import generate_all_figures


## 1. Data preparation


In [ ]:
dataset_rows = []
for name in DATASETS:
    X, y, predictors = load_dataset(name)
    dataset_rows.append({
        "dataset": name,
        "observations": len(y),
        "predictors": X.shape[1],
        "predictor_names": ", ".join(predictors),
    })

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)


## 2. Fit OLS and LAD on the empirical datasets


In [ ]:
original_metrics, original_coefficients = run_original_data()
display(original_metrics[["dataset", "model", "SSE", "SAE"]])
display(original_coefficients)


## 3. Controlled large-response-error experiment

The controlled experiment adds increasingly large response errors while keeping the predictor design fixed. SSE and SAE are evaluated against the uncontaminated responses so the fitted models can be compared against the clean relationship.


In [ ]:
contamination_metrics, contamination_shifts = run_contamination_experiment()

contamination_summary = (
    contamination_metrics
    .groupby(["contamination_fraction", "model"], as_index=False)[["SSE", "SAE"]]
    .mean()
)
display(contamination_summary)


## 4. Error-distribution experiment


In [ ]:
distribution_results = run_error_distribution_experiment()

distribution_summary = (
    distribution_results
    .groupby(["distribution", "model"], as_index=False)
    [["SSE", "SAE", "coefficient_error_l2"]]
    .median()
)
display(distribution_summary)


## 5. Runtime benchmark


In [ ]:
runtime_results = run_runtime_benchmark()

runtime_summary = (
    runtime_results
    .groupby(["n", "p", "model"], as_index=False)["runtime_seconds"]
    .median()
)
display(runtime_summary)


## 6. Solver validation


In [ ]:
validation_results = validate_lad_solver()
display(validation_results)


## 7. Focused visualizations

Only the four figures that directly support the main OLS–LAD comparison are retained.


In [ ]:
generate_all_figures()

key_figures = [
    "ols_lad_contaminated_fit.png",
    "sse_vs_contamination.png",
    "sae_vs_contamination.png",
    "hbk_multivariate_inlier_outlier.png",
]

for filename in key_figures:
    path = FIGURE_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))


## 8. Generated files


In [ ]:
result_files = sorted(path.name for path in RESULT_DIR.glob("*"))
figure_files = sorted(path.name for path in FIGURE_DIR.glob("*"))

display(pd.DataFrame({"result_files": pd.Series(result_files)}))
display(pd.DataFrame({"figure_files": pd.Series(figure_files)}))


## 9. Summary

The empirical and controlled experiments compare OLS and LAD with the same observations and predictors. The final figures focus on the central distinction between squared-error and absolute-error minimization: how the fitted lines react to large response errors, how SSE and SAE change as contamination increases, and how the two regressions behave on the HBK dataset with its supplied case groups.
